# B2: Status Classification

---

## Overview

Classify projects into standardized status categories.

**Status Categories:**
- Proposed
- In Review
- Approved
- Appealed
- Permitted
- Under Construction
- Completed
- Stalled

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import json

# Add modules to path
sys.path.insert(0, str(Path.cwd().parent.parent))

from modules.timeline_calculator import (
    classify_project_status,
    PROJECT_STATUSES,
    project_status_summary
)
from modules.data_loader import load_csv

# Configuration
with open('../../config/berkeley_config.json') as f:
    CONFIG = json.load(f)

DATA_DIR = Path(CONFIG['paths']['data_dir'])

print("Status classification mappings:")
for category, keywords in PROJECT_STATUSES.items():
    print(f"  {category}: {keywords}")

## 2. Load Projects

In [ ]:
# Load housing projects
housing_path = DATA_DIR / 'housing_projects_FINAL.csv'
df = load_csv(housing_path)

if df is not None:
    print(f"Loaded {len(df)} projects")
    
    # Show raw status values
    print(f"\nUnique status values:")
    for status in df['status'].unique():
        count = len(df[df['status'] == status])
        print(f"  {status}: {count}")

## 3. Classify All Projects

In [ ]:
# Apply classification
if df is not None:
    df['status_category'] = df['status'].apply(classify_project_status)
    
    # Summary
    summary = project_status_summary(df, 'status')
    print("Status Category Summary:")
    print("="*50)
    display(summary)
    
    # Total units by status
    if 'net_units' in df.columns:
        units_by_status = df.groupby('status_category')['net_units'].sum().sort_values(ascending=False)
        print(f"\nUnits by status category:")
        for status, units in units_by_status.items():
            print(f"  {status}: {units:,.0f} units")

## 4. Status Mapping Validation

In [ ]:
# Show mapping for each raw status
if df is not None:
    print("Status Mapping Validation:")
    print("="*60)
    
    mapping = df[['status', 'status_category']].drop_duplicates().sort_values('status_category')
    
    for _, row in mapping.iterrows():
        print(f"  '{row['status']}' -> {row['status_category']}")
    
    # Check for unknowns
    unknowns = df[df['status_category'] == 'unknown']
    if len(unknowns) > 0:
        print(f"\nWARNING: {len(unknowns)} projects with unknown status")
        print(unknowns['status'].value_counts())

## 5. Status Distribution by Year

In [ ]:
# Status by year
if df is not None and 'year' in df.columns:
    pivot = pd.pivot_table(
        df,
        index='year',
        columns='status_category',
        values='net_units',
        aggfunc='sum',
        fill_value=0
    )
    
    print("Units by Year and Status:")
    display(pivot)

## 6. Export Classified Data

In [ ]:
# Export with status categories
if df is not None:
    output_path = DATA_DIR / 'housing_projects_classified.csv'
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

---

## Summary

This notebook:
- Classified raw status values into standard categories
- Validated status mappings
- Analyzed status distribution by year

**Next:** Run `B3_progress_indicators.ipynb` to track construction progress.